# Домашняя работа №1.1 (Pandas) - максимум 3 балла
В городе SQL произошло убийство! SQL Murder Mystery - это одновременно и самостоятельный урок для изучения концепций и команд SQL, и увлекательная игра для опытных пользователей SQL, в которой нужно раскрыть интригующее преступление.
Произошло преступление, и детективу нужна ваша помощь. Детектив дал вам отчет о месте преступления, но вы каким-то образом потеряли его. Вы смутно помните, что преступление было убийством, произошедшим 15 января 2018 года, и что оно произошло в SQL City. Начните с поиска соответствующего отчета о месте преступления в базе данных полицейского управления.

In [2]:
import pandas as pd

## Условия
Главное условие - решение полностью с использованием Pandas

Данные не на русском языке, так что где-то придется пользоваться переводчиком

## Критерии оценивания

1. Нашел преступника? - 4 балла
2. Использование сортировки или срезов по данным (оператор []) во время поиска - 1 балл
3. Использование фильтрации таблиц - 2 балла
4. Использование мерджей - 3 балла


## Решение

"Начните с поиска соответствующего отчета о месте преступления в базе данных полицейского управления"

звучит как табличка `crime_scene_report`

In [3]:
crime_scene_report = pd.read_csv('data_pandas/crime_scene_report.csv')
crime_scene_report.head()

,date,type,description,city
0,20180115,robbery,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
1,20180115,murder,Life? Dont talk to me about life.,Albany
2,20180115,murder,"Mama, I killed a man, put a gun against his he...",Reno
3,20180215,murder,REDACTED REDACTED REDACTED,SQL City
4,20180215,murder,Someone killed the guard! He took an arrow to ...,SQL City


Можно заметить, дату, тип преступления, описание и город - мы знаем, что:
- *Убиство* произошедшим *15 января 2018 года*, и что оно произошло в *SQL City*
или 
- `type == marder`, `date = 20180115`, `cit == SQL City`

In [4]:
crime_scene_report['date'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 1228 entries, 0 to 1227
Series name: date
Non-Null Count  Dtype
--------------  -----
1228 non-null   int64
dtypes: int64(1)
memory usage: 9.7 KB


In [5]:
our_report = crime_scene_report[
    (crime_scene_report['type'] == 'murder') &
    (crime_scene_report['date'] == 20180115) &
    (crime_scene_report['city'] == 'SQL City')
]
our_report

,date,type,description,city
1227,20180115,murder,Security footage shows that there were 2 witne...,SQL City


In [6]:
description_text = our_report['description'].iloc[0]
description_text

'Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".'

Мы получили рапорт, давайте прочитаем его:
- Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".

Или по-русски:
- Камеры наблюдения показывают, что было 2 свидетеля. Первый свидетель живет в последнем доме на "Нортвестерн Др." (Northwestern Dr.). Второй свидетель, по имени Аннабель, живет где-то на "Франклин Авеню" (Franklin Ave.).

Такс! Я посмотрел что находится в файлах - у нас есть два свидетеля, для каждого из них указано где они живут - тогда можно выяснить как их зовут! Информация о проживании находится в таблице `person`

In [74]:
person = pd.read_csv('data_pandas/person.csv')
person.head()

,id,name,license_id,address_number,address_street_name,ssn
0,10000,Christoper Peteuil,993845,624,Bankhall Ave,747714076
1,10007,Kourtney Calderwood,861794,2791,Gustavus Blvd,477972044
2,10010,Muoi Cary,385336,741,Northwestern Dr,828638512
3,10016,Era Moselle,431897,1987,Wood Glade St,614621061
4,10025,Trena Hornby,550890,276,Daws Hill Way,223877684


Первый живет в ПОСЛЕДНЕМ ДОМЕ на "Нортвестерн Др." -  можно получить вот так

In [50]:
first_witness = person[
    person['address_street_name'] == 'Northwestern Dr'
].sort_values('address_number', ascending=False).iloc[0]
first_witness

id                               14887
name                    Morty Schapiro
license_id                      118009
address_number                    4919
address_street_name    Northwestern Dr
ssn                          111564949
Name: 499, dtype: object

Второй живет в где-то на "Франклин Авеню" (Его зовут Анабель - это нам поможет) -  можно получить вот так

In [51]:
second_witness = person[
    ((person['address_street_name'] == 'Franklin Ave') &
    person['name'].str.startswith('Annabel'))].iloc[0]
second_witness

id                              16371
name                   Annabel Miller
license_id                     490173
address_number                    103
address_street_name      Franklin Ave
ssn                         318771143
Name: 665, dtype: object

Итак, мы нашли двух сведетелей, надо найти в таблицах, что каждый из них видел - такая информация находится в папке с допросами `interview`


In [73]:
interview = pd.read_csv('data_pandas/interview.csv')
interview.head()

,person_id,transcript
0,28508,‘I deny it!’ said the March Hare.\n
1,63713,\n
2,86208,"way, and the whole party swam to the shore.\n"
3,35267,"lessons in here? Why, there’s hardly room for ..."
4,33856,\n


In [58]:
witnesses = pd.DataFrame([first_witness, second_witness]).reset_index(drop=True)
witnesses

,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949
1,16371,Annabel Miller,490173,103,Franklin Ave,318771143


Смержим с допросами

In [76]:
person_with_interview = person.merge(
    interview,
    left_on='id',
    right_on='person_id',
    how='left'
)
person_with_interview.head()

,id,name,license_id,address_number,address_street_name,ssn,person_id,transcript
0,10000,Christoper Peteuil,993845,624,Bankhall Ave,747714076,NaN,NaN
1,10007,Kourtney Calderwood,861794,2791,Gustavus Blvd,477972044,10007.0,CHAPTER IV. The Rabbit Sends in a Little Bill\n
2,10010,Muoi Cary,385336,741,Northwestern Dr,828638512,NaN,NaN
3,10016,Era Moselle,431897,1987,Wood Glade St,614621061,10016.0,\n
4,10025,Trena Hornby,550890,276,Daws Hill Way,223877684,10025.0,\n


In [60]:
witnesses_with_interview = witnesses.merge(
    interview,
    left_on='id',
    right_on='person_id',
    how='left'
)
witnesses_with_interview

,id,name,license_id,address_number,address_street_name,ssn,person_id,transcript
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949,14887,I heard a gunshot and then saw a man run out. ...
1,16371,Annabel Miller,490173,103,Franklin Ave,318771143,16371,"I saw the murder happen, and I recognized the ..."


In [62]:
print(witnesses_with_interview['transcript'].iloc[0])
print(witnesses_with_interview['transcript'].iloc[1])

I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".
I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.


Давайте почитаем что сказали Morty Schapiro и Annabel Miller:
- Morty Schapiro: I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".
- Annabel Miller: I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.

Или на русском:
- Morty Schapiro: Я услышал выстрел, а потом увидел, как выбежал мужчина. У него была сумка "Get Fit Now Gym". Номер членства на сумке начинался с "48Z". Только золотые члены имеют такие сумки. Мужчина сел в машину, номер которой содержал "H42W"
- Annabel Miller: Я видела, как произошло убийство, и узнала убийцу из своего спортзала, когда я тренировалась на прошлой неделе, 9 января

Это очень информативный допрос, я бы начал с машины - она уникальная зацепка: давайте найдем все у кого машина содержит "H42W". У нас как раз есть таблица водительских удостоверений `drivers_license`

In [65]:
drivers_license = pd.read_csv('data_pandas/drivers_license.csv')
drivers_license.head()

,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
0,100280,72,57,brown,red,male,P24L4U,Acura,MDX
1,100460,63,72,brown,brown,female,XF02T6,Cadillac,SRX
2,101029,62,74,green,green,female,VKY5KR,Scion,xB
3,101198,43,54,amber,brown,female,Y5NZ08,Nissan,Rogue
4,101255,18,79,blue,grey,female,5162Z1,Lexus,GS


In [ ]:
cars = drivers_license[drivers_license['plate_number'].str.contains('H42W', case=False, na=False)]
cars

,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
915,183779,21,65,blue,blonde,female,H42W0X,Toyota,Prius
3529,423327,30,70,brown,brown,male,0H42W2,Chevrolet,Spark LS
6240,664760,21,71,black,black,male,4H42WR,Nissan,Altima


Неудивительно, что таких машин несколько - далее стоит заметить, что по словам Morty Schapiro из спортзала выбежал мужчина - значит у нас два подозреваемых:

In [89]:
suspect = cars[cars['gender'] == 'male'].merge(
    person_with_interview,
    left_on='id',
    right_on='license_id',
    how='left'
)
suspect

,id_x,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model,id_y,name,license_id,address_number,address_street_name,ssn,person_id,transcript
0,423327,30,70,brown,brown,male,0H42W2,Chevrolet,Spark LS,67318,Jeremy Bowers,423327,530,"Washington Pl, Apt 3A",871539279,67318.0,I was hired by a woman with a lot of money. I ...
1,664760,21,71,black,black,male,4H42WR,Nissan,Altima,51739,Tushar Chandra,664760,312,Phi St,137882671,NaN,NaN


На данный момент два подозреваемых:
- Jeremy Bowers
- Tushar Chandra

У одного из них уже есть данные о допросе - он уже признался, что его наняли для убиства, но давайте проверим, посещал ли он клуб - в этом нам поможет таблица `gym_members`:

In [91]:
suspect['transcript'].iloc[0]

'I was hired by a woman with a lot of money. I don\'t know her name but I know she\'s around 5\'5" (65") or 5\'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.\n'

**Jeremy Bowers**: Меня наняла женщина с большими деньгами. Я не знаю её имени, но я знаю, что её рост около 5'5" (165 см) или 5'7" (170 см). У неё рыжие волосы, и она водит Tesla Model S. Я знаю, что она посетила концерт SQL Symphony Concert 3 раза в декабре 2017 года."

In [94]:
gym_members = pd.read_csv('data_pandas/get_fit_now_member.csv')
gym_members.head()

,id,person_id,name,membership_start_date,membership_status
0,NL318,65076,Everette Koepke,20170926,gold
1,AOE21,39426,Noe Locascio,20171005,regular
2,2PN28,63823,Jeromy Heitschmidt,20180215,silver
3,0YJ24,80651,Waneta Wellard,20171206,gold
4,3A08L,32858,Mei Bianchin,20170401,silver


In [96]:
get_fit_now_check_in = pd.read_csv('data_pandas/get_fit_now_check_in.csv')
get_fit_now_check_in.head()

,membership_id,check_in_date,check_in_time,check_out_time
0,NL318,20180212,329,365
1,NL318,20170811,469,920
2,NL318,20180429,506,554
3,NL318,20180128,124,759
4,NL318,20171027,418,1019


In [103]:
suspect_gym = gym_members[
    (gym_members['person_id'].isin(suspect['id_y'])) &
    (gym_members['id'].str.startswith('48Z')) &
    (gym_members['membership_status'] == 'gold')
]
print(suspect_gym)
suspect_checkin = get_fit_now_check_in[
    (get_fit_now_check_in['membership_id'].isin(suspect_gym['id'])) &
    (get_fit_now_check_in['check_in_date'] == 20180109)
]
print(suspect_checkin)

        id  person_id           name  membership_start_date membership_status
182  48Z55      67318  Jeremy Bowers               20160101              gold
     membership_id  check_in_date  check_in_time  check_out_time
2701         48Z55       20180109           1530            1700


Все сошлось **Jeremy Bowers** - Убийца!!!!

Он говорит:
Меня наняла женщина с большими деньгами. Я не знаю её имени, но я знаю, что её рост около 5'5" (165 см) или 5'7" (170 см). У неё рыжие волосы, и она водит Tesla Model S. Я знаю, что она посетила концерт SQL Symphony Concert 3 раза в декабре 2017 года."

Так, сначала найдем таких водителей-женщин, о которых речь

In [106]:
drivers_license_with_person_with_interview = drivers_license.merge(
    person_with_interview,
    left_on='id',
    right_on='license_id',
    how='left'
)
drivers_license_with_person_with_interview

,id_x,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model,id_y,name,license_id,address_number,address_street_name,ssn,person_id,transcript
0,100280,72,57,brown,red,male,P24L4U,Acura,MDX,22757.0,Alfonzo Lighter,100280.0,2380.0,Demeo Rd,158688705.0,NaN,NaN
1,100460,63,72,brown,brown,female,XF02T6,Cadillac,SRX,84910.0,Jayme Secor,100460.0,872.0,Cooksey Circle,221254409.0,84910.0,"drunk half the bottle, she found her head pres..."
2,101029,62,74,green,green,female,VKY5KR,Scion,xB,84921.0,Cassey Boeve,101029.0,3698.0,Pottawattami Blvd,377934170.0,84921.0,"a doze; but, on being pinched by the Hatter, i..."
3,101198,43,54,amber,brown,female,Y5NZ08,Nissan,Rogue,58014.0,Logan Helde,101198.0,929.0,Gilton St,545228106.0,NaN,NaN
4,101255,18,79,blue,grey,female,5162Z1,Lexus,GS,90704.0,Shanna Zingone,101255.0,2071.0,N Village Rd,215917206.0,90704.0,"There was a long silence after this, and Alice..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10002,999923,19,77,amber,black,female,5L0ZI4,GMC,Sierra 3500,29742.0,Lang Huels,999923.0,733.0,Nootka St,386376791.0,NaN,NaN
10003,999940,71,61,green,green,male,1B8QN8,Mitsubishi,Eclipse,45295.0,Bert Buecher,999940.0,892.0,Oak Bend Rd,722731907.0,NaN,NaN
10004,999981,67,69,brown,blue,female,1684K3,Land Rover,LR2,87184.0,Deon Gottesman,999981.0,1875.0,Hayden Rowe St,355452772.0,NaN,NaN
10005,999986,49,58,green,grey,male,F8F64H,Lexus,LS,22665.0,Troy Mandino,999986.0,978.0,Deepage Dr,544494123.0,22665.0,\n


In [ ]:
suspect_car = drivers_license_with_person_with_interview[
    (drivers_license_with_person_with_interview['car_make'] == 'Tesla') &
    (drivers_license_with_person_with_interview['car_model'] == 'Model S') &
    (drivers_license_with_person_with_interview['hair_color'] == 'red') &
    (drivers_license_with_person_with_interview['gender'] == 'female') &
    (drivers_license_with_person_with_interview['height'].between(65, 70))
]
suspect_car

,id_x,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model,id_y,name,license_id,address_number,address_street_name,ssn,person_id,transcript
1105,202298,68,66,green,red,female,500123,Tesla,Model S,99716.0,Miranda Priestly,202298.0,1883.0,Golden Ave,987756388.0,NaN,NaN
2054,291182,65,66,blue,red,female,08CM64,Tesla,Model S,90700.0,Regina George,291182.0,332.0,Maple Ave,337169072.0,NaN,NaN
9078,918773,48,65,black,red,female,917UU3,Tesla,Model S,78881.0,Red Korb,918773.0,107.0,Camerata Dr,961388910.0,NaN,NaN


Три кандидата - давайте посмотрим, кто из них 3 раза был на концерте:

In [117]:
event = pd.read_csv('data_pandas/facebook_event_checkin.csv')
print(event[event['event_name'] == 'SQL Symphony Concert'])
event.head()

       person_id  event_id            event_name      date
11         97207      1143  SQL Symphony Concert  20170322
12         97207      1143  SQL Symphony Concert  20180226
109        92433      1143  SQL Symphony Concert  20170821
110        92433      1143  SQL Symphony Concert  20180327
296        88952      1143  SQL Symphony Concert  20170918
...          ...       ...                   ...       ...
20003      24556      1143  SQL Symphony Concert  20171224
20006      99716      1143  SQL Symphony Concert  20171206
20007      99716      1143  SQL Symphony Concert  20171212
20008      99716      1143  SQL Symphony Concert  20171229
20010      67318      1143  SQL Symphony Concert  20171206

[212 rows x 4 columns]


,person_id,event_id,event_name,date
0,28508,5880,Nudists are people who wear one-button suits.\n,20170913
1,63713,3865,but that's because it's the best book on anyth...,20171009
2,63713,3999,"If Murphy's Law can go wrong, it will.\n",20170502
3,63713,6436,Old programmers never die. They just branch t...,20170926
4,82998,4470,Help a swallow land at Capistrano.\n,20171022


In [122]:
symphony = event[event['event_name'] == 'SQL Symphony Concert']
symphony_dec2017 = symphony[
    (symphony['date'] >= 20171201) &
    (symphony['date'] <= 20171231)
]

visits = symphony_dec2017.groupby('person_id').size().reset_index(name='visit_count')
visits.head()

,person_id,visit_count
0,11173,1
1,19260,1
2,19292,1
3,24397,1
4,24556,3


Кто из подозрительных водителей посещал в декабре 2017го SQL симфонию 3 раза?

In [129]:
womans = suspect_car.merge(
    visits,
    left_on='id_y',
    right_on='person_id',
    how='left'
)
womans

,id_x,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model,id_y,name,license_id,address_number,address_street_name,ssn,person_id_x,transcript,person_id_y,visit_count
0,202298,68,66,green,red,female,500123,Tesla,Model S,99716.0,Miranda Priestly,202298.0,1883.0,Golden Ave,987756388.0,NaN,NaN,99716.0,3.0
1,291182,65,66,blue,red,female,08CM64,Tesla,Model S,90700.0,Regina George,291182.0,332.0,Maple Ave,337169072.0,NaN,NaN,NaN,NaN
2,918773,48,65,black,red,female,917UU3,Tesla,Model S,78881.0,Red Korb,918773.0,107.0,Camerata Dr,961388910.0,NaN,NaN,NaN,NaN


Видим, что только `Miranda Priestly` посещала концерт 3 раза - она главный подозреваемый!!!!

Также ее проверим высокий доход

In [127]:
income = pd.read_csv('data_pandas/income.csv')
income.head()

,ssn,annual_income
0,100009868,52200
1,100169584,64500
2,100300433,74400
3,100355733,35900
4,100366269,73000


In [137]:
income.merge(
    womans,
    left_on='ssn',
    right_on='ssn',
    how='right'
)

,ssn,annual_income,id_x,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model,id_y,name,license_id,address_number,address_street_name,person_id_x,transcript,person_id_y,visit_count
0,987756388.0,310000.0,202298,68,66,green,red,female,500123,Tesla,Model S,99716.0,Miranda Priestly,202298.0,1883.0,Golden Ave,NaN,NaN,99716.0,3.0
1,337169072.0,NaN,291182,65,66,blue,red,female,08CM64,Tesla,Model S,90700.0,Regina George,291182.0,332.0,Maple Ave,NaN,NaN,NaN,NaN
2,961388910.0,278000.0,918773,48,65,black,red,female,917UU3,Tesla,Model S,78881.0,Red Korb,918773.0,107.0,Camerata Dr,NaN,NaN,NaN,NaN


У Miranda Priestly действительно очень большой доход

Все подтвердилось - дело расскрыто

### "Дело раскрыто! Убийца — Jeremy Bowers, он был нанят Miranda Priestly